<a href="https://colab.research.google.com/github/reeshyk/en-es-swap-pipeline/blob/main/IW_Work_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading the data from the sentiment analysis dataset and isolating the Spanish and English Reviews  

In [ ]:
# @title
!pip install "datasets<2.20"

from huggingface_hub import login
from datasets import load_dataset
from google.colab import drive
drive.mount('/content/drive')
login()

import datasets
import pandas as pd
from datasets import load_dataset

dataset = load_dataset("Brand24/mms", split="train")

# Filter for english and spanish only
filtered = dataset.filter(lambda x: [lang in ["en", "es"] and dom == "reviews"
               for lang, dom in zip(x["language"], x["domain"])],
    batched=True)
filtered = filtered.select_columns(["text", "_id", "label", "language", "domain", "original_dataset"])

# Collect results into a list or dataframe
df = pd.DataFrame(filtered)

# Words between 10 - 20
df["word_count"] = df["text"].str.split().str.len()
df = df[(df["word_count"] >= 10) & (df["word_count"] <= 20)]

# saving full dataset to drive
df.to_parquet("/content/drive/MyDrive/mms_filtered.parquet", index=False)

# Then split by language
df_es = df[df["language"] == "es"]
df_en = df[df["language"] == "en"]

print(f"Spanish: {len(df_es)} rows")
print(f"English: {len(df_en)} rows")


Focus only on sentences with 10 - 20 words

In [ ]:
# @title
df_en_sample = df_en.groupby("label").sample(n=3000, random_state=42)
df_es_sample = df_es.groupby("label").sample(n=3000, random_state=42)

# saving each to drive
df_en_sample.to_parquet("/content/drive/MyDrive/mms_en_sample.parquet", index=False)
df_es_sample.to_parquet("/content/drive/MyDrive/mms_es_sample.parquet", index=False)

print(f"English sample: {len(df_en_sample)} rows")
print(df_en_sample["label"].value_counts())

print(f"\nSpanish sample: {len(df_es_sample)} rows")
print(df_es_sample["label"].value_counts())

# ***Download Each dataset from drive***

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

df = pd.read_parquet("/content/drive/MyDrive/mms_filtered.parquet")
df_en_sample = pd.read_parquet("/content/drive/MyDrive/mms_en_sample.parquet")
df_es_sample = pd.read_parquet("/content/drive/MyDrive/mms_es_sample.parquet")

print(f"Full filtered: {len(df)} rows")
print(f"English sample: {len(df_en_sample)} rows")
print(f"Spanish sample: {len(df_es_sample)} rows")

Mounted at /content/drive
Full filtered: 404801 rows
English sample: 9000 rows
Spanish sample: 9000 rows


# Build Vocabularies for each dataset

In [ ]:
!wget https://dl.fbaipublicfiles.com/arrival/dictionaries/en-es.txt -O en-es.txt
!wget https://dl.fbaipublicfiles.com/arrival/dictionaries/es-en.txt -O es-en.txt
!pip install scikit-learn
!pip install spacy
!pip install triton --upgrade
!pip install transformers --upgrade
!python -m spacy download en_core_web_md
!python -m spacy download es_core_news_md
!pip install wordfreq

import os
import torch
import spacy
import pickle
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    AutoTokenizer,
    AutoModelForMaskedLM,
    pipeline
)
from wordfreq import top_n_list, word_frequency

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

nlp_en = spacy.load("en_core_web_md")
nlp_es = spacy.load("es_core_news_md")

def load_muse_dictionary(path):
    pairs = {}
    with open(path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            src, tgt = parts
            src, tgt = src.lower(), tgt.lower()
            if src != tgt and src.isalpha() and tgt.isalpha():
                if src not in pairs:
                    pairs[src] = []
                pairs[src].append(tgt)
    print(f"Loaded {len(pairs)} source words from {path}")
    return pairs

print("Loading MUSE dictionaries...")
en_to_es_dict = load_muse_dictionary("en-es.txt")
es_to_en_dict = load_muse_dictionary("es-en.txt")

--2026-04-02 18:47:52--  https://dl.fbaipublicfiles.com/arrival/dictionaries/en-es.txt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.35.37.111, 13.35.37.84, 13.35.37.90, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.35.37.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1891373 (1.8M) [text/x-c++]
Saving to: ‘en-es.txt’

en-es.txt           100%[===================>]   1.80M  --.-KB/s    in 0.01s   

2026-04-02 18:47:52 (182 MB/s) - ‘en-es.txt’ saved [1891373/1891373]

--2026-04-02 18:47:52--  https://dl.fbaipublicfiles.com/arrival/dictionaries/es-en.txt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.35.37.111, 13.35.37.84, 13.35.37.90, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.35.37.111|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1891536 (1.8M) [text/plain]
Saving to: ‘es-en.txt’

es-en.txt           100%[===================>]   1.80M  --.-KB/

Building Vocabulary Using SpaCy medium models

In [ ]:
model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2", device=device)

# Load translation models
print("Loading translation models...")
en_es_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
en_es_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-en-es").to(device)

es_en_tokenizer = MarianTokenizer.from_pretrained("Helsinki-NLP/opus-mt-es-en")
es_en_model = MarianMTModel.from_pretrained("Helsinki-NLP/opus-mt-es-en").to(device)

print("Loading MLM model...")
mlm_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
mlm_model = AutoModelForMaskedLM.from_pretrained("bert-base-multilingual-cased").to(device)


from functools import lru_cache

@lru_cache(maxsize=10000)
def translate_sentence(sentence, source_lang="en"):
    if source_lang == "en":
        tokenizer, trans_model = en_es_tokenizer, en_es_model
    else:
        tokenizer, trans_model = es_en_tokenizer, es_en_model

    inputs = tokenizer([sentence], return_tensors="pt",
                       padding=True, truncation=True, max_length=512).to(device)
    with torch.no_grad():
        translated = trans_model.generate(**inputs)
    return tokenizer.decode(translated[0], skip_special_tokens=True)


def get_best_replacement(sentence, token, target_lemma, source_lang="en"):
    target_lang = "es" if source_lang == "en" else "en"
    target_nlp = nlp_es if target_lang == "es" else nlp_en

    if token.pos_ == "VERB":
        translated_sentence = translate_sentence(sentence, source_lang)
        target_doc = target_nlp(translated_sentence)

        for t in target_doc:
            if t.pos_ == "VERB" and t.lemma_.lower() == target_lemma:
                return t.text.lower()
        return target_lemma
    else:
        return target_lemma


def score_in_context(sentence, original_word, replacement):
    masked = sentence.replace(original_word, mlm_tokenizer.mask_token, 1)
    inputs = mlm_tokenizer(masked, return_tensors="pt",
                           truncation=True, max_length=512).to(device)

    mask_idx = (inputs["input_ids"] == mlm_tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
    if len(mask_idx) == 0:
        return float("-inf")

    with torch.no_grad():
        outputs = mlm_model(**inputs)

    logits = outputs.logits[0, mask_idx[0]]
    probs = torch.log_softmax(logits, dim=-1)

    token_id = mlm_tokenizer.convert_tokens_to_ids(replacement)
    if token_id == mlm_tokenizer.unk_token_id:
        return float("-inf")

    return probs[token_id].item()


def verify_semantic_similarity(source_word, replacement, threshold=0.4):
    embeddings = model.encode([source_word, replacement], convert_to_numpy=True)
    similarity = float(cosine_similarity([embeddings[0]], [embeddings[1]])[0][0])
    return similarity >= threshold, similarity


def get_position(token, doc):
    for sent in doc.sents:
        if token.i >= sent.start and token.i < sent.end:
            sent_length = sent.end - sent.start
            relative_pos = token.i - sent.start
            third = sent_length / 3
            if relative_pos < third:
                return "beginning"
            elif relative_pos < third * 2:
                return "middle"
            else:
                return "end"
    return "middle"


def swap_word(sentence, source_lang="en", pos=None, position=None):
    dictionary = en_to_es_dict if source_lang == "en" else es_to_en_dict
    nlp = nlp_en if source_lang == "en" else nlp_es
    target_lang = "es" if source_lang == "en" else "en"

    doc = nlp(sentence)

    if pos:
        eligible = [token for token in doc if token.pos_ == pos.upper()]
    else:
        eligible = [token for token in doc if token.pos_ in ["NOUN", "ADJ", "VERB"]]

    if position:
        eligible = [token for token in eligible
                    if get_position(token, doc) == position.lower()]

    if not eligible:
        return "no_eligible"

    token = eligible[0]
    lookup = token.lemma_.lower() if token.pos_ == "VERB" else token.text.lower()

    if lookup not in dictionary:
        return "no_candidates"

    translations = dictionary[lookup]

    for target_lemma in translations:
        if target_lemma == lookup:
            continue

        target_freq = word_frequency(target_lemma, target_lang)
        source_freq = word_frequency(target_lemma, source_lang)
        if target_freq < source_freq:
            continue

        replacement = get_best_replacement(sentence, token, target_lemma, source_lang)

        context_score = score_in_context(sentence, token.text, replacement)
        if context_score == float("-inf"):
            context_score = score_in_context(sentence, token.text, target_lemma)
            if context_score == float("-inf"):
                continue
            replacement = target_lemma

        is_similar, similarity = verify_semantic_similarity(token.text, replacement)
        if not is_similar:
            continue

        swapped = sentence.replace(token.text, replacement, 1)
        return {
            "sentence": swapped,
            "original_word": token.text,
            "replacement": replacement,
            "similarity": similarity,
            "context_score": context_score
        }

    return "no_candidates"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading translation models...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading MLM model...


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-multilingual-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Building the architecture for word swapping

Swap Using word mappings

In [ ]:
# Test it
print(swap_word("I make food", source_lang="en", pos="VERB"))
print(swap_word("Excellent saw. I bought for cutting up fallen trees and exercise. Tones the brazos nicely.", source_lang="en", pos="VERB", position="middle"))
print(swap_word("El café estaba absolutamente terrible", source_lang="es", pos="NOUN"))

{'sentence': 'I hacer food', 'original_word': 'make', 'replacement': 'hacer', 'similarity': 0.7580811977386475, 'context_score': -12.895367622375488}
{'sentence': 'Excellent véase. I bought for cutting up fallen trees and exercise. Tones the brazos nicely.', 'original_word': 'saw', 'replacement': 'véase', 'similarity': 0.413573294878006, 'context_score': -19.82491111755371}
{'sentence': 'El coffee estaba absolutamente terrible', 'original_word': 'café', 'replacement': 'coffee', 'similarity': 0.9662539958953857, 'context_score': -13.443241119384766}


Testing the Swapping function

In [ ]:
def test_swap_words(df, source_lang="en", n=10, seed=42):
    """Test swap_word on a random sample of sentences."""
    sample = df.sample(n=n, random_state=seed)

    for idx, row in sample.iterrows():
        sentence = row["text"]
        print(f"\n{'='*60}")
        print(f"Original: {sentence}")
        print(f"Label: {row['label']}")
        print("-" * 60)

        for pos in ["NOUN", "ADJ", "VERB"]:
            for position in ["beginning", "middle", "end"]:
                result = swap_word(sentence, source_lang=source_lang,
                                   pos=pos, position=position)

                if result == "no_eligible":
                    print(f"{pos:<6} {position:<12} → no eligible token")
                elif result == "no_candidates":
                    print(f"{pos:<6} {position:<12} → no candidates found")
                else:
                    print(f"{pos:<6} {position:<12} → '{result['original_word']}' "
                          f"replaced with '{result['replacement']}' "
                          f"(similarity: {result['similarity']:.3f})")
                    print(f"{'':20} {result['sentence']}")

# Test on English
test_swap_words(df_en_sample, source_lang="en", n=10)

# Test on Spanish
test_swap_words(df_es_sample, source_lang="es", n=10)


Original: Works well , or at least as well as the original one I received with my camera.
Label: 2
------------------------------------------------------------
NOUN   beginning    → no eligible token
NOUN   middle       → 'one' replaced with 'una' (similarity: 0.933)
                     Works well , or at least as well as the original una I received with my camera.
NOUN   end          → 'camera' replaced with 'cámara' (similarity: 0.968)
                     Works well , or at least as well as the original one I received with my cámara.
ADJ    beginning    → no candidates found
ADJ    middle       → 'original' replaced with 'originales' (similarity: 0.961)
                     Works well , or at least as well as the originales one I received with my camera.
ADJ    end          → no eligible token
VERB   beginning    → 'Works' replaced with 'obra' (similarity: 0.624)
                     obra well , or at least as well as the original one I received with my camera.
VERB   middle      

Analyzing Sentences

In [ ]:
!pip install transformers
from transformers import pipeline

# Load multilingual sentiment model
sentiment_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
    device=0 if torch.cuda.is_available() else -1
)

def get_sentiment_score(text):
    """Returns sentiment as a score from 1-5."""
    try:
        result = sentiment_model(text[:512])[0]  # truncate to model max length
        # Model returns '1 star' to '5 stars' as labels
        score = int(result["label"].split()[0])
        return score, result["score"]  # score 1-5, confidence
    except:
        return None, None


def analyze_dataset(df, source_lang="en"):
    all_results = []

    # First pass - collect all sentences and swaps
    print("Generating swaps...")
    rows_data = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        sentence = row["text"]
        row_data = {"original_text": sentence, "label": row["label"]}
        row_data["no_eligible_count"] = 0
        row_data["no_candidates_count"] = 0

        for pos in ["NOUN", "ADJ", "VERB"]:
            for position in ["beginning", "middle", "end"]:
                swap_result = swap_word(sentence, source_lang=source_lang,
                                       pos=pos, position=position)
                key = f"{pos}_{position}"
                if swap_result == "no_eligible":
                    row_data["no_eligible_count"] += 1
                    row_data[f"{key}_sentence"] = None
                elif swap_result == "no_candidates":
                    row_data["no_candidates_count"] += 1
                    row_data[f"{key}_sentence"] = None
                else:
                    row_data[f"{key}_sentence"] = swap_result["sentence"]
                    row_data[f"{key}_original_word"] = swap_result["original_word"]
                    row_data[f"{key}_replacement"] = swap_result["replacement"]
                    row_data[f"{key}_similarity"] = swap_result["similarity"]

        rows_data.append(row_data)

    # Collect all sentences that need sentiment analysis
    print("Collecting sentences for sentiment analysis...")
    all_sentences = []
    for row_data in rows_data:
        all_sentences.append(row_data["original_text"])
        for pos in ["NOUN", "ADJ", "VERB"]:
            for position in ["beginning", "middle", "end"]:
                key = f"{pos}_{position}"
                if row_data.get(f"{key}_sentence") is not None:
                    all_sentences.append(row_data[f"{key}_sentence"])

    # Second pass - batch sentiment analysis
    print(f"Running sentiment analysis on {len(all_sentences)} sentences...")
    sentiment_results = sentiment_model(
        all_sentences,
        batch_size=64,
        truncation=True,
        max_length=512
    )

    # Map sentences to their sentiment scores
    sentiment_map = {}
    for sentence, result in zip(all_sentences, sentiment_results):
        score = int(result["label"].split()[0])
        sentiment_map[sentence] = (score, result["score"])

    # Third pass - assemble final results
    print("Assembling results...")
    for row_data in rows_data:
        orig_score, orig_conf = sentiment_map[row_data["original_text"]]
        row_data["original_score"] = orig_score
        row_data["original_confidence"] = orig_conf

        for pos in ["NOUN", "ADJ", "VERB"]:
            for position in ["beginning", "middle", "end"]:
                key = f"{pos}_{position}"
                swapped_sentence = row_data.get(f"{key}_sentence")
                if swapped_sentence is not None:
                    swap_score, swap_conf = sentiment_map[swapped_sentence]
                    row_data[f"{key}_score"] = swap_score
                    row_data[f"{key}_confidence"] = swap_conf
                    row_data[f"{key}_score_diff"] = swap_score - orig_score

        all_results.append(row_data)

    return pd.DataFrame(all_results)


# Run on both datasets
print("Analyzing English dataset...")
df_en_results = analyze_dataset(df_en_sample, source_lang="en")

print("\nAnalyzing Spanish dataset...")
df_es_results = analyze_dataset(df_es_sample, source_lang="es")

# Save results
df_en_results.to_parquet("/content/drive/MyDrive/en_sentiment_results.parquet", index=False)
df_es_results.to_parquet("/content/drive/MyDrive/es_sentiment_results.parquet", index=False)

print(df_en_results.head())
print(df_es_results.head())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Analyzing English dataset...
Generating swaps...


100%|██████████| 9000/9000 [1:00:56<00:00,  2.46it/s]


Running sentiment analysis on 36799 sentences...
Assembling results...

Analyzing Spanish dataset...
Generating swaps...


 82%|████████▏ | 7393/9000 [39:49<10:23,  2.58it/s]

Print Swap Analysis

In [ ]:
# Reload in future sessions without rerunning analysis
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

df_en_results = pd.read_parquet("/content/drive/MyDrive/en_sentiment_results.parquet")
df_es_results = pd.read_parquet("/content/drive/MyDrive/es_sentiment_results.parquet")

print(f"English results: {len(df_en_results)} rows")
print(f"Spanish results: {len(df_es_results)} rows")


def print_swap_analysis(df_results, lang="en"):
    positions = ["beginning", "middle", "end"]
    pos_tags = ["NOUN", "ADJ", "VERB"]

    print(f"\n{'='*60}")
    print(f"Swap Analysis - {lang.upper()}")
    print(f"{'='*60}\n")

    print(f"{'':15}", end="")
    for position in positions:
        print(f"{position:>20}", end="")
    print()
    print("-" * 75)

    for pos in pos_tags:
        print(f"{pos:<15}", end="")
        for position in positions:
            key = f"{pos}_{position}"
            sentence_col = f"{key}_sentence"
            diff_col = f"{key}_score_diff"

            if sentence_col not in df_results.columns:
                print(f"{'N/A':>20}", end="")
                continue

            successful_swaps = df_results[sentence_col].notna()
            total_successful = successful_swaps.sum()

            if total_successful == 0:
                print(f"{'0/0':>20}", end="")
                continue

            sentiment_shifted = (
                df_results.loc[successful_swaps, diff_col].notna() &
                (df_results.loc[successful_swaps, diff_col] != 0)
            ).sum()

            # Format fraction as a single string first, then pad
            fraction = f"{sentiment_shifted}/{total_successful}"
            print(f"{fraction:>20}", end="")
        print()

    print("-" * 75)
    total_no_eligible = df_results["no_eligible_count"].sum()
    total_no_candidates = df_results["no_candidates_count"].sum()
    print(f"\nTotal skipped - no eligible token: {total_no_eligible}")
    print(f"Total skipped - no candidates:     {total_no_candidates}")
    print()

print_swap_analysis(df_en_results, lang="English")
print_swap_analysis(df_es_results, lang="Spanish")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
English results: 9000 rows
Spanish results: 9000 rows

Swap Analysis - ENGLISH

                          beginning              middle                 end
---------------------------------------------------------------------------
NOUN                       209/3139            155/3387            153/3501
ADJ                        461/2993            238/2445            170/1896
VERB                       519/4536            292/3558            172/2344
---------------------------------------------------------------------------

Total skipped - no eligible token: 35690
Total skipped - no candidates:     17511


Swap Analysis - SPANISH

                          beginning              middle                 end
---------------------------------------------------------------------------
NOUN                       248/4109            199/4304            196/43

Analyze specific cases where there was a sentiment shift

In [ ]:
# Add direction parameter
def show_sentiment_shifts(df_results, n=10, min_diff=1,
                          pos_filter=None, direction=None):
    """
    Show examples of sentences where a sentiment shift occurred.
    min_diff: minimum absolute score difference to show
    """
    pos_tags = ["NOUN", "ADJ", "VERB"]
    positions = ["beginning", "middle", "end"]

    examples = []

    for idx, row in df_results.iterrows():
        for pos in pos_tags:
            for position in positions:
                key = f"{pos}_{position}"
                diff_col = f"{key}_score_diff"
                sentence_col = f"{key}_sentence"

                if diff_col not in df_results.columns:
                    continue
                if pd.isna(row[diff_col]) or pd.isna(row[sentence_col]):
                    continue
                if abs(row[diff_col]) >= min_diff:
                  if direction == "negative" and row[diff_col] >= 0:
                      continue
                  if direction == "positive" and row[diff_col] <= 0:
                      continue
                  if pos_filter and pos != pos_filter:
                      continue
                  examples.append({
                        "original": row["original_text"],
                        "swapped": row[sentence_col],
                        "original_word": row[f"{key}_original_word"],
                        "replacement": row[f"{key}_replacement"],
                        "original_score": row["original_score"],
                        "swapped_score": row[f"{key}_score"],
                        "score_diff": row[diff_col],
                        "pos": pos,
                        "position": position
                  })

    # Sort by absolute score diff
    examples = sorted(examples, key=lambda x: abs(x["score_diff"]), reverse=True)

    # Print top n
    for ex in examples[:n]:
        print(f"\n{'='*60}")
        print(f"POS: {ex['pos']} | Position: {ex['position']} | "
              f"Diff: {int(ex['score_diff']):+d}")
        print(f"Original  ({int(ex['original_score'])} stars): {ex['original']}")
        print(f"Swapped   ({int(ex['swapped_score'])} stars): {ex['swapped']}")
        print(f"Swap: '{ex['original_word']}' → '{ex['replacement']}'")

# Show top 10 largest shifts
print("=== NEGATIVE ENGLISH SENTIMENT SHIFTS ===")
# Only show cases where sentiment got worse
show_sentiment_shifts(df_en_results, n=10, min_diff=1, direction="negative")

print("\n=== POSITIVE ENGLISH SENTIMENT SHIFTS ===")
# Only show cases where sentiment got worse
show_sentiment_shifts(df_en_results, n=10, min_diff=1, direction="positive")

print("\n=== NEGATIVE SPANISH SENTIMENT SHIFTS ===")
show_sentiment_shifts(df_es_results, n=10, min_diff=1, direction="negative")

print("\n=== POSITIVE SPANISH SENTIMENT SHIFTS ===")
show_sentiment_shifts(df_es_results, n=10, min_diff=1, direction="positive")

=== NEGATIVE ENGLISH SENTIMENT SHIFTS ===

POS: VERB | Position: beginning | Diff: -4
Original  (5 stars): It was like 5 pages long. They met and five months later he came for her and that is all!
Swapped   (1 stars): It was like 5 pages long. They conocer and five months later he came for her and that is all!
Swap: 'met' → 'conocer'

POS: NOUN | Position: middle | Diff: -4
Original  (5 stars): Received open box item. Amazon has been doing this a lot lately. Inspect everything!
Swapped   (1 stars): Received open caja item. Amazon has been doing this a lot lately. Inspect everything!
Swap: 'box' → 'caja'

POS: VERB | Position: middle | Diff: -4
Original  (5 stars): Soulless and    even more damning    virtually joyless, XXX achieves near virtuosity in its crapulence.
Swapped   (1 stars): Soulless and    even more damning    virtually joyless, XXX logra near virtuosity in its crapulence.
Swap: 'achieves' → 'logra'

POS: NOUN | Position: beginning | Diff: -4
Original  (5 stars): Top thing

Analysis Across 6 Metrics

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd
from tqdm import tqdm

# Load LaBSE for semantic similarity (better for multilingual)
labse_model = SentenceTransformer("sentence-transformers/LaBSE", device=device)

# ================================================================
# METRIC 1: Flip Rate (FR)
# ================================================================
def compute_flip_rate(df_results, lang=""):
    pos_tags = ["NOUN", "ADJ", "VERB"]
    positions = ["beginning", "middle", "end"]
    flip_rates = {}
    combos = [(pos, position) for pos in pos_tags for position in positions]

    for pos, position in tqdm(combos, desc=f"Flip Rate {lang}"):
        key = f"{pos}_{position}"
        diff_col = f"{key}_score_diff"
        sentence_col = f"{key}_sentence"

        if sentence_col not in df_results.columns:
            continue

        swapped = df_results[sentence_col].notna()
        total = swapped.sum()
        if total == 0:
            continue

        flipped = (df_results.loc[swapped, diff_col] != 0).sum()
        flip_rates[key] = {
            "flipped": int(flipped),
            "total": int(total),
            "rate": float(flipped / total)
        }

    return flip_rates


# ================================================================
# METRIC 2: Minimum Swap Distance to Flip (MSDF)
# ================================================================
# def compute_msdf(df, source_lang="en", max_swaps=9):
#     ...


# ================================================================
# METRIC 3: Semantic Similarity (SemSim)
# ================================================================
def compute_semsim(df_results, lang=""):
    pos_tags = ["NOUN", "ADJ", "VERB"]
    positions = ["beginning", "middle", "end"]
    semsim_results = {}
    combos = [(pos, position) for pos in pos_tags for position in positions]

    for pos, position in tqdm(combos, desc=f"SemSim {lang}"):
        key = f"{pos}_{position}"
        sentence_col = f"{key}_sentence"

        if sentence_col not in df_results.columns:
            continue

        valid = df_results[df_results[sentence_col].notna()]
        if len(valid) == 0:
            continue

        orig_sentences = valid["original_text"].tolist()
        swap_sentences = valid[sentence_col].tolist()

        orig_embeddings = labse_model.encode(orig_sentences,
                                              batch_size=64,
                                              convert_to_numpy=True,
                                              show_progress_bar=False)
        swap_embeddings = labse_model.encode(swap_sentences,
                                              batch_size=64,
                                              convert_to_numpy=True,
                                              show_progress_bar=False)

        sims = [float(util.cos_sim(o, s))
                for o, s in zip(orig_embeddings, swap_embeddings)]

        diff_col = f"{key}_score_diff"
        high_sim_mask = np.array(sims) >= 0.8
        flipped = (valid[diff_col].values != 0)

        flip_at_high_sim = (high_sim_mask & flipped).sum()
        total_high_sim = high_sim_mask.sum()

        semsim_results[key] = {
            "mean_similarity": float(np.mean(sims)),
            "flip_rate_at_high_sim": float(flip_at_high_sim / total_high_sim)
                                      if total_high_sim > 0 else 0.0,
            "total_high_sim": int(total_high_sim)
        }

    return semsim_results


# ================================================================
# METRIC 4: Robustness Gap
# ================================================================
def compute_robustness_gap(df_results, lang=""):
    pos_tags = ["NOUN", "ADJ", "VERB"]
    positions = ["beginning", "middle", "end"]
    label_map = {0: 1, 1: 3, 2: 5}

    def is_correct(pred_score, true_label):
        true_score = label_map.get(true_label)
        if true_score is None:
            return False
        def to_class(s):
            if s <= 2: return "negative"
            elif s == 3: return "neutral"
            else: return "positive"
        return to_class(pred_score) == to_class(true_score)

    orig_correct = df_results.apply(
        lambda r: is_correct(r["original_score"], r["label"]), axis=1
    ).mean()

    gaps = {}
    combos = [(pos, position) for pos in pos_tags for position in positions]

    for pos, position in tqdm(combos, desc=f"Robustness Gap {lang}"):
        key = f"{pos}_{position}"
        score_col = f"{key}_score"
        sentence_col = f"{key}_sentence"

        if score_col not in df_results.columns:
            continue

        valid = df_results[df_results[sentence_col].notna()].copy()
        if len(valid) == 0:
            continue

        swap_correct = valid.apply(
            lambda r: is_correct(r[score_col], r["label"]), axis=1
        ).mean()

        gaps[key] = {
            "original_accuracy": float(orig_correct),
            "swapped_accuracy": float(swap_correct),
            "robustness_gap": float(orig_correct - swap_correct)
        }

    return gaps


# ================================================================
# METRIC 5: Flip Rate by Language Mixing Index (LMI)
# ================================================================
def compute_lmi(sentence, lang="en"):
    words = sentence.lower().split()
    if not words:
        return 0.0

    other_lang = "es" if lang == "en" else "en"
    other_count = sum(
        1 for w in words
        if word_frequency(w, other_lang) > word_frequency(w, lang)
    )
    return other_count / len(words)

def compute_flip_rate_by_lmi(df_results, source_lang="en", n_bins=5):
    pos_tags = ["NOUN", "ADJ", "VERB"]
    positions = ["beginning", "middle", "end"]
    lmi_flip_data = []
    combos = [(pos, position) for pos in pos_tags for position in positions]

    for pos, position in tqdm(combos, desc=f"LMI {source_lang.upper()}"):
        key = f"{pos}_{position}"
        sentence_col = f"{key}_sentence"
        diff_col = f"{key}_score_diff"

        if sentence_col not in df_results.columns:
            continue

        valid = df_results[df_results[sentence_col].notna()].copy()
        if len(valid) == 0:
            continue

        valid["lmi"] = valid[sentence_col].apply(
            lambda s: compute_lmi(s, lang=source_lang)
        )
        valid["flipped"] = valid[diff_col] != 0
        valid["swap_key"] = key
        lmi_flip_data.append(valid[["lmi", "flipped", "swap_key"]])

    if not lmi_flip_data:
        return None

    combined = pd.concat(lmi_flip_data)
    combined["lmi_bin"] = pd.cut(combined["lmi"], bins=n_bins)
    lmi_summary = combined.groupby("lmi_bin")["flipped"].agg(["mean", "count"])
    lmi_summary.columns = ["flip_rate", "count"]
    return lmi_summary


# ================================================================
# METRIC 6: Per-Language-Pair Asymmetry
# ================================================================
def compute_asymmetry(df_en_results, df_es_results):
    pos_tags = ["NOUN", "ADJ", "VERB"]
    positions = ["beginning", "middle", "end"]
    asymmetry = {}
    combos = [(pos, position) for pos in pos_tags for position in positions]

    for pos, position in tqdm(combos, desc="Asymmetry EN↔ES"):
        key = f"{pos}_{position}"
        diff_col = f"{key}_score_diff"
        sentence_col = f"{key}_sentence"

        if sentence_col in df_en_results.columns:
            en_valid = df_en_results[df_en_results[sentence_col].notna()]
            en_flips = (en_valid[diff_col] != 0).sum()
            en_total = len(en_valid)
            en_rate = en_flips / en_total if en_total > 0 else 0
        else:
            en_rate, en_total = 0, 0

        if sentence_col in df_es_results.columns:
            es_valid = df_es_results[df_es_results[sentence_col].notna()]
            es_flips = (es_valid[diff_col] != 0).sum()
            es_total = len(es_valid)
            es_rate = es_flips / es_total if es_total > 0 else 0
        else:
            es_rate, es_total = 0, 0

        asymmetry[key] = {
            "en_to_es_flip_rate": float(en_rate),
            "es_to_en_flip_rate": float(es_rate),
            "asymmetry": float(en_rate - es_rate)
        }

    return asymmetry


# ================================================================
# RUN ALL METRICS
# ================================================================
print("Computing Flip Rates...")
fr_en = compute_flip_rate(df_en_results, lang="EN")
fr_es = compute_flip_rate(df_es_results, lang="ES")

print("\nComputing Semantic Similarity...")
semsim_en = compute_semsim(df_en_results, lang="EN")
semsim_es = compute_semsim(df_es_results, lang="ES")

print("\nComputing Robustness Gap...")
gap_en = compute_robustness_gap(df_en_results, lang="EN")
gap_es = compute_robustness_gap(df_es_results, lang="ES")

print("\nComputing LMI Flip Rates...")
lmi_en = compute_flip_rate_by_lmi(df_en_results, source_lang="en")
lmi_es = compute_flip_rate_by_lmi(df_es_results, source_lang="es")

print("\nComputing Asymmetry...")
asymmetry = compute_asymmetry(df_en_results, df_es_results)

print("\nDone!")

# print("\nComputing MSDF (this may take a while)...")
# msdf_en = compute_msdf(df_en_sample, source_lang="en")
# msdf_es = compute_msdf(df_es_sample, source_lang="es")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing Flip Rates...


Flip Rate ES: 100%|██████████| 9/9 [00:00<00:00, 977.69it/s]



Computing Semantic Similarity...


...


Computing LMI Flip Rates...


LMI EN: 100%|██████████| 9/9 [00:00<00:00, 12.88it/s]
/tmp/ipykernel_1407/581625881.py:195: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  lmi_summary = combined.groupby("lmi_bin")["flipped"].agg(["mean", "count"])
LMI ES: 100%|██████████| 9/9 [00:00<00:00, 12.91it/s]
/tmp/ipykernel_1407/581625881.py:195: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  lmi_summary = combined.groupby("lmi_bin")["flipped"].agg(["mean", "count"])



Computing Asymmetry...


Asymmetry EN↔ES: 100%|██████████| 9/9 [00:00<00:00, 156.17it/s]


Done!


Print Results Summary

In [ ]:
def print_all_metrics(fr, semsim, gap, lmi, asymmetry, lang="en"):
    print(f"\n{'='*60}")
    print(f"FULL METRICS REPORT - {lang.upper()}")
    print(f"{'='*60}")

    print(f"\n--- Flip Rate ---")
    for key, val in fr.items():
        print(f"{key:<20} {val['flipped']}/{val['total']} "
              f"({val['rate']*100:.1f}%)")

    print(f"\n--- Semantic Similarity & Flip Rate at High SemSim ---")
    for key, val in semsim.items():
        print(f"{key:<20} mean_sim={val['mean_similarity']:.3f} "
              f"flip@sim>=0.8={val['flip_rate_at_high_sim']*100:.1f}% "
              f"(n={val['total_high_sim']})")

    print(f"\n--- Robustness Gap ---")
    for key, val in gap.items():
        print(f"{key:<20} orig_acc={val['original_accuracy']*100:.1f}% "
              f"swap_acc={val['swapped_accuracy']*100:.1f}% "
              f"gap={val['robustness_gap']*100:.1f}%")

    print(f"\n--- LMI Flip Rate ---")
    print(lmi)

    print(f"\n--- Asymmetry (en->es vs es->en) ---")
    for key, val in asymmetry.items():
        print(f"{key:<20} en->es={val['en_to_es_flip_rate']*100:.1f}% "
              f"es->en={val['es_to_en_flip_rate']*100:.1f}% "
              f"diff={val['asymmetry']*100:+.1f}%")

    # print(f"\n--- MSDF ---")
    # print(f"Mean min swaps to flip: {msdf['min_swaps_to_flip'].mean():.2f}")
    # print(f"Never flipped: {msdf['min_swaps_to_flip'].isna().sum()} sentences")
    # print(msdf['min_swaps_to_flip'].value_counts().sort_index())

print_all_metrics(fr_en, semsim_en, gap_en, lmi_en, asymmetry, lang="en")
print_all_metrics(fr_es, semsim_es, gap_es, lmi_es, asymmetry, lang="es")


FULL METRICS REPORT - EN

--- Flip Rate ---
NOUN_beginning       209/3139 (6.7%)
NOUN_middle          155/3387 (4.6%)
NOUN_end             153/3501 (4.4%)
ADJ_beginning        461/2993 (15.4%)
ADJ_middle           238/2445 (9.7%)
ADJ_end              170/1896 (9.0%)
VERB_beginning       519/4536 (11.4%)
VERB_middle          292/3558 (8.2%)
VERB_end             172/2344 (7.3%)

--- Semantic Similarity & Flip Rate at High SemSim ---
NOUN_beginning       mean_sim=0.979 flip@sim>=0.8=6.7% (n=3139)
NOUN_middle          mean_sim=0.983 flip@sim>=0.8=4.6% (n=3387)
NOUN_end             mean_sim=0.983 flip@sim>=0.8=4.4% (n=3501)
ADJ_beginning        mean_sim=0.975 flip@sim>=0.8=15.4% (n=2993)
ADJ_middle           mean_sim=0.983 flip@sim>=0.8=9.7% (n=2445)
ADJ_end              mean_sim=0.984 flip@sim>=0.8=9.0% (n=1896)
VERB_beginning       mean_sim=0.976 flip@sim>=0.8=11.4% (n=4535)
VERB_middle          mean_sim=0.984 flip@sim>=0.8=8.2% (n=3558)
VERB_end             mean_sim=0.984 flip@sim>=0.8=